# Wagner Questionnaire Demo

This notebook loads the local Wagner package, asks questions interactively, and scores the response.

Run cells in order:
1. Cell 2: Load package
2. Cell 3: Interactive questionnaire (`y/yes` or `n/no`)
3. Cell 4: Score and label

In [ ]:
const path = require('path');

function loadWagnerModule() {
  const candidates = [
    '.',
    './src/index.js',
    './collection/wagner',
    './collection/wagner/src/index.js'
  ];

  for (const candidate of candidates) {
    try {
      return require(path.resolve(candidate));
    } catch (error) {
      // Try next candidate path.
    }
  }

  throw new Error('Could not load Wagner module from expected paths.');
}

const { runQuestionnaire, analyzeQuestionnaireResponse, QUESTIONS } = loadWagnerModule();

console.log(`Loaded Wagner module with ${QUESTIONS.length} questions.`);

In [ ]:
const readline = require('readline');

function promptYesNo(promptText) {
  return new Promise((resolve) => {
    const rl = readline.createInterface({
      input: process.stdin,
      output: process.stdout
    });

    const ask = () => {
      rl.question(`${promptText} [y/n]: `, (raw) => {
        const value = String(raw || '').trim().toLowerCase();
        if (value === 'y' || value === 'yes') {
          rl.close();
          resolve(true);
          return;
        }
        if (value === 'n' || value === 'no') {
          rl.close();
          resolve(false);
          return;
        }

        console.log("Please answer with 'y'/'yes' or 'n'/'no'.");
        ask();
      });
    };

    ask();
  });
}

async function askYesNo(question) {
  console.log(`\n${question.id}: ${question.text}`);

  const definitionEntries = Object.entries(question.definitions || {});
  if (definitionEntries.length > 0) {
    console.log('Definitions:');
    for (const [term, info] of definitionEntries) {
      console.log(`- ${term}: ${info.definition}`);
    }
  }

  return promptYesNo('Your answer');
}

(async () => {
  console.log('Starting interactive Wagner questionnaire...');
  globalThis.questionnaireOutcome = await runQuestionnaire(askYesNo);
  console.log('\nQuestionnaire outcome:');
  console.log(JSON.stringify(globalThis.questionnaireOutcome, null, 2));
})();

In [2]:
if (!globalThis.questionnaireOutcome) {
  throw new Error('Run Cell 3 first to generate questionnaireOutcome.');
}

const analysis = analyzeQuestionnaireResponse(globalThis.questionnaireOutcome);

console.log('Scored analysis:');
console.log(JSON.stringify(analysis, null, 2));

if (analysis.wagner_score.length > 0) {
  console.log(`Final Wagner score: ${analysis.wagner_score[0]}`);
  console.log(`Label: ${analysis.grade_label[0]}`);
}